# S08 — Primary four-product operator sensitivity

Assesses the stability of the primary four-product benchmark under the tested footprint and pixel extraction operators.

The public copy is output-stripped; authoritative exported tables and figures are distributed separately in the repository.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math

import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import transform as crs_transform
from rasterio.windows import Window
from shapely.geometry import Point, box
from IPython.display import display, Markdown

PROJECT = Path(r"C:\Users\Dell\Desktop\Publication_Clarck\Natural_Sampling")
OUT = PROJECT / "Ablations" / "PreSubmission_Independent_Robustness" / "results" / "primary_g4_operator_sensitivity_v1"
TABLES = OUT / "tables"
CACHE_DIR = OUT / "cache"
TABLES.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_CACHE = PROJECT / "Results" / "Final_Article_Harmonized_GEDIAnchored_NaturalP1" / "CHM_Comparison" / "GEDI_TEST_product_valid_support_signed_errors.csv.gz"
RUN_EXTRACTION = True
MIN_FOOTPRINT_COVERAGE = 0.80
FOOTPRINT_RADIUS_M = 12.5
EXPECTED_N = {"Ifran": 5053, "Maamoura": 1796, "Agadir": 6401}
PRODUCT_CACHE_NAMES = {
    "Our model": "Our B4 Phase 2",
    "Pa24": "Pauls 2020",
    "L23": "Lang 2020",
    "T24": "Meta/Tolan 2023",
}
PRODUCT_ORDER = list(PRODUCT_CACHE_NAMES)

PRODUCT_ROOT = PROJECT / "CHM_Products_Comparison"
SITES = {
    "Ifran": {"key": "Ifran_6"},
    "Maamoura": {"key": "Maamoura"},
    "Agadir": {"key": "Agadir"},
}
for forest, cfg in SITES.items():
    pauls_tag = "EPSG32630" if forest == "Ifran" else "EPSG32629"
    root = PRODUCT_ROOT / cfg["key"]
    cfg["maps"] = {
        "L23": root / f"ETH_Lang_2020_CHM_10m/clean/mosaic/{cfg['key']}__ETH_Lang_2020_CHM_10m__clean__EPSG32630.tif",
        "T24": root / f"Meta_WRI_Tolan_2023_CHM_resampled_10m/clean/mosaic/{cfg['key']}__Meta_WRI_Tolan_2023_CHM_resampled_10m__clean__EPSG32630.tif",
        "Pa24": PRODUCT_ROOT / f"_PAULS_2020_VERIFIED_V1/{forest}/Pauls_et_al_2024_CHM_2020_10m/clean/mosaic/{forest}__Pauls_et_al_2024_CHM_2020_10m__clean__{pauls_tag}.tif",
    }

assert SOURCE_CACHE.is_file(), SOURCE_CACHE
raw = pd.read_csv(SOURCE_CACHE, dtype={"shot_id": str})
raw["shot_id"] = raw["shot_id"].astype(str)

# Reconstruct exactly the primary G4 shot-ID intersection used by Table 3/Figs. 8-11.
support_blocks = []
for forest, expected in EXPECTED_N.items():
    sf = raw[raw["forest"].eq(forest) & raw["product"].isin(PRODUCT_CACHE_NAMES.values())].copy()
    valid_sets = {
        label: set(sf.loc[sf["product"].eq(cache_name) & sf["prediction"].notna(), "shot_id"])
        for label, cache_name in PRODUCT_CACHE_NAMES.items()
    }
    common = set.intersection(*(valid_sets[p] for p in PRODUCT_ORDER))
    if len(common) != expected:
        raise RuntimeError(f"{forest}: reconstructed primary G4 support n={len(common)}, expected n={expected}")
    coords = (sf[sf["shot_id"].isin(common)]
              .sort_values(["shot_id", "product"])
              .drop_duplicates("shot_id")
              [["forest", "shot_id", "rh95", "gedi_year", "lon", "lat"]])
    support_blocks.append(coords)

primary_points = pd.concat(support_blocks, ignore_index=True)
observed = primary_points.groupby("forest")["shot_id"].nunique().to_dict()
assert observed == EXPECTED_N, (observed, EXPECTED_N)
primary_points.to_csv(TABLES / "01_primary_g4_shot_registry.csv.gz", index=False, compression="gzip")
display(Markdown("### Exact primary G4 support recovered"))
display(pd.DataFrame({"expected_n": EXPECTED_N, "recovered_n": observed}))


In [ ]:
def valid_value(values, dataset):
    values = np.asarray(values, dtype=float)
    valid = np.isfinite(values)
    if dataset.nodata is not None and np.isfinite(dataset.nodata):
        valid &= values != dataset.nodata
    valid &= values != -9999
    return values, valid


def sample_two_operators(path, frame, radius=FOOTPRINT_RADIUS_M):
    rows = []
    with rasterio.open(path) as ds:
        xs, ys = crs_transform("EPSG:4326", ds.crs, frame["lon"].tolist(), frame["lat"].tolist())
        for rec, x, y in zip(frame.itertuples(index=False), xs, ys):
            row, col = ds.index(x, y)
            centre = np.nan
            footprint_mean = np.nan
            coverage = 0.0
            if 0 <= row < ds.height and 0 <= col < ds.width:
                value, valid = valid_value(ds.read(1, window=Window(col, row, 1, 1)), ds)
                if valid.any():
                    centre = float(value[valid][0])

                footprint = Point(x, y).buffer(radius, resolution=32)
                pixel_areas, pixel_values = [], []
                valid_area = 0.0
                row_buffer = int(math.ceil(radius / abs(ds.transform.e))) + 2
                col_buffer = int(math.ceil(radius / abs(ds.transform.a))) + 2
                for rr in range(max(0, row-row_buffer), min(ds.height, row+row_buffer+1)):
                    for cc in range(max(0, col-col_buffer), min(ds.width, col+col_buffer+1)):
                        x0, y0 = ds.xy(rr, cc, offset="ul")
                        x1, y1 = ds.xy(rr, cc, offset="lr")
                        intersection_area = footprint.intersection(
                            box(min(x0, x1), min(y0, y1), max(x0, x1), max(y0, y1))
                        ).area
                        if intersection_area <= 0:
                            continue
                        value, valid = valid_value(ds.read(1, window=Window(cc, rr, 1, 1)), ds)
                        if valid.any():
                            pixel_areas.append(intersection_area)
                            pixel_values.append(float(value[valid][0]))
                            valid_area += intersection_area
                coverage = valid_area / footprint.area
                if pixel_areas and coverage >= MIN_FOOTPRINT_COVERAGE:
                    footprint_mean = float(np.average(pixel_values, weights=pixel_areas))
            rows.append((str(rec.shot_id), centre, footprint_mean, coverage))
    return pd.DataFrame(rows, columns=["shot_id", "centre_pixel", "footprint_area_weighted", "footprint_coverage"])


def our_annual_path(forest, year):
    ecosystem = {"Ifran": "Dense", "Maamoura": "Low_Sparsity", "Agadir": "Sparse"}[forest]
    filename = f"{forest}_B4_C15_Phase2_Y{year}_M05-09_T4_SLIDING_2018-2025_ENSEMBLE.tif"
    return PROJECT / "Inference_Harmonized_GEDIAnchored_NaturalP1" / ecosystem / forest / "Phase2" / f"Y{year}" / "Annual" / filename


LONG_CACHE = CACHE_DIR / "02_primary_g4_all_products_two_operators.csv.gz"
if RUN_EXTRACTION:
    blocks = []
    for forest, cfg in SITES.items():
        forest_points = primary_points[primary_points["forest"].eq(forest)].copy()
        for product in PRODUCT_ORDER:
            pieces = []
            if product == "Our model":
                for year, year_points in forest_points.groupby("gedi_year"):
                    raster_path = our_annual_path(forest, int(year))
                    if not raster_path.is_file():
                        raise FileNotFoundError(raster_path)
                    pieces.append(sample_two_operators(raster_path, year_points))
            else:
                raster_path = cfg["maps"][product]
                if not raster_path.is_file():
                    raise FileNotFoundError(raster_path)
                pieces.append(sample_two_operators(raster_path, forest_points))
            sampled = pd.concat(pieces, ignore_index=True)
            sampled["forest"] = forest
            sampled["product"] = product
            blocks.append(forest_points[["shot_id", "rh95"]].merge(sampled, on="shot_id", how="left", validate="one_to_one"))
            print(f"[DONE] {forest} - {product}", flush=True)
    wide = pd.concat(blocks, ignore_index=True)
    long = wide.melt(
        id_vars=["forest", "product", "shot_id", "rh95", "footprint_coverage"],
        value_vars=["centre_pixel", "footprint_area_weighted"],
        var_name="operator",
        value_name="prediction",
    )
    long.to_csv(LONG_CACHE, index=False, compression="gzip")
else:
    long = pd.read_csv(LONG_CACHE, dtype={"shot_id": str})

support = long.groupby(["forest", "operator", "product"])["prediction"].agg(n="count", missing=lambda x: x.isna().sum()).reset_index()
display(support)
for forest, expected in EXPECTED_N.items():
    sf = support[support["forest"].eq(forest)]
    if not ((sf["n"] == expected).all() and (sf["missing"] == 0).all()):
        bad = sf[(sf["n"] != expected) | (sf["missing"] != 0)]
        raise RuntimeError(
            f"{forest}: at least one product/operator is incomplete on the exact primary support. "
            "Do not silently shrink the population. Inspect the displayed rows:\n" + bad.to_string(index=False)
        )


In [ ]:
def regression_metrics(frame):
    y = pd.to_numeric(frame["rh95"], errors="coerce").to_numpy(float)
    p = pd.to_numeric(frame["prediction"], errors="coerce").to_numpy(float)
    valid = np.isfinite(y) & np.isfinite(p)
    y, p = y[valid], p[valid]
    error = p - y
    return {
        "n": len(y),
        "mae": np.mean(np.abs(error)),
        "rmse": np.sqrt(np.mean(error**2)),
        "bias": np.mean(error),
        "r2": 1 - np.sum(error**2) / np.sum((y-y.mean())**2),
        "r": np.corrcoef(y, p)[0, 1],
        "slope": np.polyfit(y, p, 1)[0],
        "std_ratio": np.std(p) / np.std(y),
    }


rows = []
for (forest, operator, product), group in long.groupby(["forest", "operator", "product"], sort=False):
    rows.append({"forest": forest, "operator": operator, "product": product, **regression_metrics(group)})
metrics = pd.DataFrame(rows)
metrics["product"] = pd.Categorical(metrics["product"], PRODUCT_ORDER, ordered=True)
metrics["rank_mae"] = metrics.groupby(["forest", "operator"])["mae"].rank(method="min")
metrics["rank_rmse"] = metrics.groupby(["forest", "operator"])["rmse"].rank(method="min")
metrics = metrics.sort_values(["forest", "operator", "product"])
metrics.to_csv(TABLES / "02_primary_g4_operator_metrics_exact.csv", index=False)
display(metrics.style.format({k: "{:.6f}" for k in ["mae", "rmse", "bias", "r2", "r", "slope", "std_ratio"]}))

ranking_rows = []
for (forest, operator), group in metrics.groupby(["forest", "operator"], sort=False):
    mae_order = " < ".join(group.sort_values(["mae", "product"])["product"].astype(str))
    rmse_order = " < ".join(group.sort_values(["rmse", "product"])["product"].astype(str))
    ranking_rows.append({"forest": forest, "operator": operator, "mae_order": mae_order, "rmse_order": rmse_order})
rankings = pd.DataFrame(ranking_rows)
rankings.to_csv(TABLES / "03_primary_g4_operator_rankings.csv", index=False)
display(Markdown("### Non-rounded product rankings"))
display(rankings)

stability = []
for forest, group in rankings.groupby("forest"):
    stability.append({
        "forest": forest,
        "mae_ranking_unchanged": group["mae_order"].nunique() == 1,
        "rmse_ranking_unchanged": group["rmse_order"].nunique() == 1,
    })
stability = pd.DataFrame(stability)
stability.to_csv(TABLES / "04_primary_g4_operator_stability_decision.csv", index=False)
display(Markdown("### Stability decision"))
display(stability)


In [ ]:
compact = metrics[["forest", "operator", "product", "n", "mae", "rmse"]].copy()
compact["MAE / RMSE (m)"] = compact.apply(lambda x: f"{x.mae:.2f} / {x.rmse:.2f}", axis=1)
table = compact.pivot(index=["forest", "operator"], columns="product", values="MAE / RMSE (m)").reset_index()
table = table[["forest", "operator", *PRODUCT_ORDER]]
table.to_csv(TABLES / "05_table_C3_primary_G4_ready.csv", index=False)
display(table)

lines = []
for forest in EXPECTED_N:
    for operator in ["centre_pixel", "footprint_area_weighted"]:
        row = table[(table["forest"] == forest) & (table["operator"] == operator)].iloc[0]
        lines.append(
            f"{forest} & {operator.replace('_', ' ')} & "
            + " & ".join(str(row[p]) for p in PRODUCT_ORDER)
            + r" \\"
        )
(TABLES / "06_table_C3_latex_rows.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")

all_mae_stable = bool(stability["mae_ranking_unchanged"].all())
all_rmse_stable = bool(stability["rmse_ranking_unchanged"].all())
if all_mae_stable and all_rmse_stable:
    wording = "Operator choice did not change the MAE or RMSE product ranking on the primary four-product GEDI support in any landscape."
elif all_mae_stable:
    wording = "Operator choice did not change the MAE ranking or the principal benchmark conclusion on the primary four-product GEDI support; minor RMSE-order changes are reported explicitly in Table C.3."
else:
    wording = "At least one MAE ranking changed with the extraction operator; operator-specific rankings must be reported without a support-invariant claim."

(TABLES / "07_recommended_manuscript_wording.txt").write_text(wording + "\n", encoding="utf-8")
manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "source_cache": str(SOURCE_CACHE),
    "source_cache_sha256": hashlib.sha256(SOURCE_CACHE.read_bytes()).hexdigest(),
    "expected_primary_support": EXPECTED_N,
    "products": PRODUCT_ORDER,
    "operators": ["centre_pixel", "footprint_area_weighted"],
    "footprint_radius_m": FOOTPRINT_RADIUS_M,
    "minimum_valid_footprint_coverage": MIN_FOOTPRINT_COVERAGE,
    "run_extraction": RUN_EXTRACTION,
}
(OUT / "reproducibility_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

display(Markdown("### Recommended manuscript wording"))
print(wording)
print(f"\nOutputs written to: {OUT}")
